# 연속 user 메시지 보내기 — Claude Code 스타일 컨텍스트 주입

Claude Code 소스를 보면 user 메시지를 2~3개 묶어서 보내는 패턴이 자주 나옵니다.
이게 가능한 이유는 Claude API의 규칙 때문입니다:

> **같은 role의 메시지가 연속으로 오면, API가 하나의 턴(turn)으로 합쳐서 처리한다.**

규칙 정리:
- 첫 메시지는 반드시 `user`
- `user` → `user` → `user` 연속 허용 (한 턴으로 결합됨)
- 에러가 나는 건 role 규칙 위반이 아니라 첫 메시지가 `assistant`인 경우 등

**왜 이렇게 쓰나?** 실제 사용자가 타이핑한 입력과, 하네스(앱)가 끼워넣는 컨텍스트
(환경 정보, 리마인더, 파일 상태 등)를 **별도 메시지로 분리**하면 코드가 조합 가능(composable)해지고,
프롬프트 캐싱과의 궁합도 좋아집니다. 이 노트북에서 하나씩 확인합니다.

## 1. 클라이언트 초기화

In [1]:
import os

import anthropic
from dotenv import load_dotenv

load_dotenv()
assert os.environ.get("ANTHROPIC_API_KEY"), ".env 파일에 ANTHROPIC_API_KEY를 설정해주세요"

client = anthropic.Anthropic()
MODEL = "claude-opus-4-8"


def text_of(response) -> str:
    return next(b.text for b in response.content if b.type == "text")


print("클라이언트 준비 완료")

클라이언트 준비 완료


## 2. 기본 확인 — user 메시지 3개가 한 턴으로 합쳐진다

서로 다른 정보를 담은 user 메시지 3개를 연속으로 보내고,
모델이 세 메시지를 전부 보고 종합해서 답하는지 확인합니다.

In [2]:
messages = [
    {"role": "user", "content": "저는 부산에 살아요."},
    {"role": "user", "content": "취미는 등산입니다."},
    {"role": "user", "content": "제가 사는 곳 근처에서 취미를 즐길 만한 장소 하나만 추천해줘. 한 문단으로."},
]

response = client.messages.create(model=MODEL, max_tokens=2048, messages=messages)
print(text_of(response))
print("\n→ 부산(메시지1) + 등산(메시지2) + 요청(메시지3)을 전부 반영했다면 한 턴으로 결합된 것")

부산에서 등산을 즐기신다면 **금정산**을 추천드려요. 부산의 진산(鎭山)으로 불리는 금정산은 해발 801미터의 고당봉을 중심으로 다양한 등산 코스가 있어 초보자부터 숙련자까지 모두 즐길 수 있습니다. 특히 사적으로 지정된 금정산성을 따라 걷는 능선길은 부산 시내와 낙동강을 한눈에 내려다볼 수 있는 탁 트인 경관을 자랑하고, 하산길에 범어사에 들러 고즈넉한 산사의 정취까지 느낄 수 있어 등산의 재미를 한층 더해줍니다. 지하철로도 접근이 편리해 부산 시민들이 즐겨 찾는 대표적인 산이니 주말 나들이로 다녀오시기 좋을 거예요.

→ 부산(메시지1) + 등산(메시지2) + 요청(메시지3)을 전부 반영했다면 한 턴으로 결합된 것


## 3. Claude Code 스타일 — 컨텍스트 주입 (system-reminder 패턴)

Claude Code가 user 메시지를 묶어 보내는 대표적인 이유가 이것입니다.
사용자가 실제로 타이핑한 입력 **앞이나 뒤에**, 하네스가 자동 생성한 컨텍스트
(환경 상태, 날짜, 리마인더)를 **별도 user 메시지**로 끼워넣습니다.

문자열을 이어붙이지 않고 메시지를 분리하는 이유:
- 사용자 입력을 변형하지 않고 원형 그대로 보존 (로깅/디버깅 용이)
- 주입 로직을 켜고 끄기 쉬움 — 리스트에 append만 하면 됨
- 이력 재구성 시 어떤 부분이 주입분인지 명확

In [ ]:
# 하네스가 자동 생성하는 컨텍스트 (사용자는 이걸 타이핑한 적 없음)
injected_context = (
    "<system-reminder>\n"
    "현재 사용자 계정 정보 (자동 주입됨):\n"
    "- 구독 등급: Pro (월 100회 요청 한도)\n"
    "- 이번 달 사용량: 87회\n"
    "- 결제일: 매월 20일\n"
    "오늘 날짜: 2026-07-15\n"
    "</system-reminder>"
)

# 사용자가 실제로 타이핑한 입력
user_typed = "나 이번 달에 요청 얼마나 남았어? 그리고 한도는 언제 리셋돼?"

messages = [
    {"role": "user", "content": injected_context},  # 주입분
    {"role": "user", "content": user_typed},        # 사용자 원본 입력
]

response = client.messages.create(
    model=MODEL,
    max_tokens=2048,
    system="당신은 SaaS 고객지원 어시스턴트입니다. <system-reminder>는 시스템이 주입한 계정 컨텍스트이며 사용자 발화가 아닙니다.",
    messages=messages,
)
print(text_of(response))
print("\n→ '13회 남음 / 20일 리셋'을 답했다면 주입 컨텍스트를 정상 활용한 것")

## 4. 빠른 연속 입력 큐잉 패턴

챗 UI에서 사용자가 답변을 기다리지 않고 메시지를 연달아 보내는 경우,
매번 API를 호출하지 않고 **큐에 쌓았다가 한 번에** 보낼 수 있습니다.
모델이 세 요청을 하나의 일관된 턴으로 처리하므로 응답 품질도 좋고 호출 비용도 1회분입니다.

In [ ]:
# 사용자가 연달아 입력한 메시지들 (아직 API 호출 안 함)
pending_queue = [
    "README 요약 좀 해줘... 아 잠깐",
    "아니다, CONTRIBUTING 가이드도 같이 봐줘",
    "그리고 둘을 비교해서 신규 기여자한테 뭐가 더 중요한지 알려줘",
]

fake_readme = "# MyLib\n빠른 JSON 파서. 설치: pip install mylib. 라이선스: MIT."
fake_contributing = "# 기여 가이드\nPR 전에 테스트 필수. 커밋 메시지는 conventional commits. 코드리뷰 2인 승인 필요."

messages = [
    {"role": "user", "content": f"<file name='README.md'>\n{fake_readme}\n</file>\n\n<file name='CONTRIBUTING.md'>\n{fake_contributing}\n</file>"},
]
# 큐에 쌓인 입력을 각각 별도 user 메시지로 추가
messages += [{"role": "user", "content": q} for q in pending_queue]

response = client.messages.create(model=MODEL, max_tokens=2048, messages=messages)
print(text_of(response))
print(f"\n→ user 메시지 {len(messages)}개 → API 호출 1번으로 처리")

## 5. 비교 — 메시지 여러 개 vs 한 메시지의 콘텐츠 블록 여러 개

같은 내용을 표현하는 방법이 두 가지 있습니다. 모델 입장에선 거의 동일한 하나의 턴입니다.

```python
# 방법 A: 연속 user 메시지 (Claude Code 스타일)
[{"role": "user", "content": "컨텍스트"}, {"role": "user", "content": "질문"}]

# 방법 B: 한 메시지 안의 여러 text 블록
[{"role": "user", "content": [{"type": "text", "text": "컨텍스트"}, {"type": "text", "text": "질문"}]}]
```

선택 기준:
- **방법 A (메시지 분리)**: 주입 시점이 서로 다르거나(큐잉), 이력 관리 코드에서 단위별로 넣고 빼야 할 때 유리
- **방법 B (블록 분리)**: 논리적으로 한 발화인데 이미지+텍스트처럼 타입이 섞이거나, 블록 단위 `cache_control`을 걸어야 할 때 필수

실행해서 두 방식의 응답이 동등한지 확인해봅니다.

In [ ]:
context = "우리 팀 규칙: 모든 답변은 존댓말, 두 문장 이내."
question = "파이썬에서 리스트와 튜플의 차이는?"

# 방법 A: 연속 메시지
resp_a = client.messages.create(
    model=MODEL, max_tokens=1024,
    messages=[
        {"role": "user", "content": context},
        {"role": "user", "content": question},
    ],
)

# 방법 B: 한 메시지, 블록 2개
resp_b = client.messages.create(
    model=MODEL, max_tokens=1024,
    messages=[
        {"role": "user", "content": [
            {"type": "text", "text": context},
            {"type": "text", "text": question},
        ]},
    ],
)

print("[방법 A — 연속 메시지]")
print(text_of(resp_a))
print("\n[방법 B — 콘텐츠 블록]")
print(text_of(resp_b))

## 6. 캐싱과의 궁합 — 왜 '수정'이 아니라 '추가'인가

프롬프트 캐싱은 접두사 매칭이므로, **기존 메시지를 수정하면 그 지점부터 캐시가 전부 깨집니다.**
반면 새 메시지를 **뒤에 추가**하면 앞의 캐시된 접두사는 그대로 유지됩니다.

```
❌ 시스템 프롬프트에 상태를 계속 갱신     → 매 요청 캐시 전체 무효화
❌ 기존 user 메시지에 컨텍스트를 이어붙임  → 그 메시지 이후 캐시 무효화
✅ 새 user 메시지로 컨텍스트를 뒤에 추가   → 기존 접두사 캐시 히트 유지
```

Claude Code가 매 턴 `<system-reminder>`를 별도의 새 user 메시지로 끼워넣는 것도 이 이유가 큽니다.
긴 대화 이력 전체가 캐시에서 0.1배 비용으로 읽히는 상태를 유지하면서, 변하는 정보만 끝에 덧붙이는 거죠.
(자세한 실험은 `claude_prompt_caching.ipynb` 참고)

## 7. (보너스) `role: "system"` 메시지 — 진짜 운영자 채널

`<system-reminder>`를 user 메시지로 넣는 방식엔 한 가지 약점이 있습니다.
user 콘텐츠에 들어가는 텍스트라서, 악의적 입력이 흉내낼 수 있다는 것(프롬프트 인젝션).

그래서 Claude Opus 4.8부터는 **대화 중간에 `role: "system"` 메시지**를 넣을 수 있습니다
(베타 헤더 불필요). 캐시 접두사를 깨지 않으면서, 위조 불가능한 운영자 권한 지시를 전달하는 채널입니다.

규칙: user 메시지 뒤에 와야 하고, 마지막 항목이거나 뒤에 assistant 턴이 와야 하며, 첫 메시지로는 불가.

In [ ]:
messages = [
    {"role": "user", "content": "프롬프트 캐싱이 뭔지 설명해줘."},
    # 운영자 지시 — 사용자 발화와 분리된 시스템 권한 채널
    {"role": "system", "content": "간결 모드가 활성화되었습니다. 3문장 이내로 답하세요."},
]

response = client.messages.create(model=MODEL, max_tokens=1024, messages=messages)
print(text_of(response))
print("\n→ 3문장 이내로 답했다면 mid-conversation system 메시지가 적용된 것")